# Classify records by descriptions

In [ ]:
!pip install -U bitsandbytes

## Import libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Dataset
from tqdm import tqdm
import pandas as pd
import os
import json
import shutil

## Initialize model

In [ ]:
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    #torch_dtype="auto",
    device_map="auto"
)


## Classify descriptions

In [ ]:
def classify_problem(description: str):
    LABELS = [
        "Array", "String", "Hash Table", "Math", "Dynamic Programming", "Sorting",
        "Greedy", "Depth-First Search", "Breadth-First Search", "Binary Search",
        "Graph Theory", "Tree", "Heap (Priority Queue)", "Two Pointers",
        "Sliding Window", "Prefix Sum", "Bit Manipulation", "Matrix", "Counting",
        "Simulation", "Stack", "Queue", "Backtracking", "Union-Find",
        "Number Theory", "Linked List", "Divide and Conquer", "Recursion",
        "Bitmask", "Combinatorics", "Design", "Shortest Path",
        "Monotonic Stack", "Monotonic Queue",
    ]

    label_text = "\n".join(LABELS)

    prompt = f"""
You are a strict classification model.

Your task:
Given the problem description below, choose EXACTLY ONE category from the allowed list.

Rules:
- You MUST output only one category.
- The output MUST match EXACTLY one item in the list (case-sensitive).
- DO NOT explain, DO NOT add punctuation, DO NOT add extra words.
- If multiple categories seem possible, choose the closest match.

Allowed Categories:
{label_text}

Problem:
{description}

Output only the category name:

"""
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=3
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response


In [ ]:
ds = load_dataset("ByteDance-Seed/Code-Contests-Plus", "default")

In [ ]:
df = ds["train"].to_pandas()    
DROP_COLS = ['validator', 'generator', 'generator_cmd', 'checker']
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

batch_size = 32  

output_file = "code_contests_plus.jsonl"

with open(output_file, "w") as f:
    
    with tqdm(total=len(df), desc="Processing Samples", unit="sample") as pbar:
        for i in range(0, len(df), batch_size):
            df_batch = df.iloc[i:i+batch_size].copy()
            df_batch['category'] = df_batch["description"].apply(classify_problem)
            
            for _, row in df_batch.iterrows():
                f.write(row.to_json() + "\n")
            
            pbar.update(len(df_batch))
            del df_batch

## Output dataset

In [ ]:
FILE_TO_UPLOAD = output_file 
DATASET_TITLE = "ByteDanceNew" 
DATASET_SLUG = "bytedance-new"
UPLOAD_FOLDER = 'dataset_upload_temp'   
user_secrets = UserSecretsClient()

os.environ['KAGGLE_USERNAME'] = 'vinhvuive'
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")

if os.path.exists(UPLOAD_FOLDER):
    shutil.rmtree(UPLOAD_FOLDER)
os.makedirs(UPLOAD_FOLDER)

if os.path.exists(FILE_TO_UPLOAD):
    shutil.move(FILE_TO_UPLOAD, f"{UPLOAD_FOLDER}/{FILE_TO_UPLOAD}")
    print(f"Copied file {FILE_TO_UPLOAD}")
else:
    raise FileNotFoundError(f"File {FILE_TO_UPLOAD} not found")

metadata = {
    "title": DATASET_TITLE,
    "id": f"{os.environ['KAGGLE_USERNAME']}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f"{UPLOAD_FOLDER}/dataset-metadata.json", 'w') as f:
    json.dump(metadata, f)

result = os.system(f"kaggle datasets create -p {UPLOAD_FOLDER} --dir-mode zip")
